In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window


spark = SparkSession.builder \
    .master("local[*]") \
    .appName("blue-owls-assessment") \
    .getOrCreate()

spark.version  # confirm Spark is running

'3.5.0'

In [6]:
fact_order_items_df = spark.read.option("header", True).option("inferSchema",True).csv("/home/jovyan/blue-owls-de-assessment/submission/output/gold/fact_order_items.csv")
fact_order_items_df.createOrReplaceTempView("fact_order_items")

dim_products_df = spark.read.option("header", True).option("inferSchema",True).csv("/home/jovyan/blue-owls-de-assessment/submission/output/gold/dim_products.csv")
dim_products_df.createOrReplaceTempView("dim_products")

In [7]:
df=spark.sql("""WITH base AS (
    SELECT 
        dp.product_category_name,
        YEAR(f.order_date) AS year,
        MONTH(f.order_date) AS month,
        (f.price + f.freight_value) AS revenue
    FROM fact_order_items f
    JOIN dim_products dp
        ON f.product_key = dp.product_key
),

-- Monthly aggregation
monthly AS (
    SELECT
        product_category_name,
        year,
        month,
        SUM(revenue) AS monthly_revenue,
        COUNT(*) AS txn_count
    FROM base
    GROUP BY product_category_name, year, month
),

-- Filter categories with at least 10 transactions
filtered AS (
    SELECT *
    FROM monthly
    WHERE txn_count >= 10
),

-- Rank categories within each month
ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY year, month
            ORDER BY monthly_revenue DESC
        ) AS monthly_rank
    FROM filtered
),

-- Top 5 categories by overall revenue
top_categories AS (
    SELECT product_category_name
    FROM ranked
    GROUP BY product_category_name
    ORDER BY SUM(monthly_revenue) DESC
    LIMIT 5
),

-- Keep only top categories
final_base AS (
    SELECT r.*
    FROM ranked r
    JOIN top_categories t
        ON r.product_category_name = t.product_category_name
),

-- Add MoM and rolling avg
final AS (
    SELECT
        product_category_name,
        year,
        month,
        monthly_revenue,
        monthly_rank,

        -- MoM Growth %
        ROUND(
            (
                (monthly_revenue - LAG(monthly_revenue) OVER (
                    PARTITION BY product_category_name
                    ORDER BY year, month
                ))
                /
                LAG(monthly_revenue) OVER (
                    PARTITION BY product_category_name
                    ORDER BY year, month
                )
            ) * 100,
            2
        ) AS mom_growth_pct,

        -- Rolling 3-month average
        ROUND(
            AVG(monthly_revenue) OVER (
                PARTITION BY product_category_name
                ORDER BY year, month
                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
            ),
            2
        ) AS rolling_3m_avg_revenue

    FROM final_base
)

SELECT *
FROM final
ORDER BY product_category_name, year, month;""")

In [8]:
df.show()

+---------------------+----+-----+------------------+------------+--------------+----------------------+
|product_category_name|year|month|   monthly_revenue|monthly_rank|mom_growth_pct|rolling_3m_avg_revenue|
+---------------------+----+-----+------------------+------------+--------------+----------------------+
|         beleza_saude|2018|    7|           97854.0|           2|          NULL|               97854.0|
|         beleza_saude|2018|    8|117450.96000000006|           1|         20.03|             107652.48|
|      cama_mesa_banho|2018|    7| 52661.01000000003|           4|          NULL|              52661.01|
|      cama_mesa_banho|2018|    8| 59277.15999999995|           4|         12.56|              55969.08|
|        esporte_lazer|2018|    7| 51982.00000000004|           5|          NULL|               51982.0|
|        esporte_lazer|2018|    8| 50152.75000000004|           5|         -3.52|              51067.38|
|   relogios_presentes|2018|    7| 98107.12000000017|  